# 7. Tables and figures

Builds the paper's tables and figures from the two backtest grids and saves the figures (PNG and PDF) to `Data/Article_figures/`.

In [ ]:
# load the two grid summaries + shared helpers
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_DIR = "Data"
BULL   = f"{DATA_DIR}/Backtest_results/results_all.parquet"          # 28,800 rows — 2023-26 test
CRISIS = f"{DATA_DIR}/Backtest_results_crisis/results_all.parquet"   # 9,600 rows  — 2008-12 crisis
FIGDIR = f"{DATA_DIR}/Article_figures"; os.makedirs(FIGDIR, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(f"{FIGDIR}/{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(f"{FIGDIR}/{name}.pdf", bbox_inches="tight")

CAP, YEARS_TEST, YEARS_CRISIS = 1e5, 3.416, 5.0
tr = lambda ec: (ec / CAP - 1) * 100
ar = lambda ec, yrs: ((ec / CAP) ** (1 / yrs) - 1) * 100

def load(path, yrs):
    d = pd.read_parquet(path)
    d["gmm"] = d["gmm"].astype(str).isin(["True", "true", "1"])
    d["TR"]  = tr(d["end_capital_validation"])
    d["AR"]  = ar(d["end_capital_validation"], yrs)
    d["MDD"] = d["mdd_validation"]
    d["CR"]  = d["AR"] / d["MDD"]
    d["oneN_TR"], d["oneN_MDD"] = tr(d["1/N_validation"]), d["1/N_mdd_validation"]
    return d

bull   = load(BULL, YEARS_TEST)
crisis = load(CRISIS, YEARS_CRISIS)
N1_TR, N1_MDD = bull["oneN_TR"].iloc[0], bull["oneN_MDD"].iloc[0]
N1_AR = ar(bull["1/N_validation"].iloc[0], YEARS_TEST); N1_CR = N1_AR / N1_MDD
print("bull", bull.shape, "| crisis", crisis.shape)
print("1/N test:  TR %.0f  AR %.1f  MDD %.1f  CR %.2f" % (N1_TR, N1_AR, N1_MDD, N1_CR))

In [ ]:
# Table 4: the four hypotheses, tested out of sample
n = len(bull); nbeat = int((bull["end_capital_validation"] > bull["1/N_validation"]).sum())
p1 = stats.binomtest(nbeat, n, 0.5).pvalue
pair = (bull.pivot_table(index=["algo", "scale", "fusion", "tier", "train_start", "x_days", "k"],
                         columns="gmm", values="end_capital_validation").dropna())
p2 = stats.wilcoxon(pair[True], pair[False]).pvalue
gate_pp = tr(pair[True]).mean() - tr(pair[False]).mean()
rho, p3 = stats.spearmanr(bull["end_capital_train"], bull["end_capital_validation"])
cmed, cN = crisis["MDD"].median(), crisis["oneN_MDD"].iloc[0]

print("H1  beat 1/N %.1f%% (%d/%d), binomial p=%.1e" % (100 * nbeat / n, nbeat, n, p1))
print("H2  regime gate %+.0f pp test return, Wilcoxon p=%.1e" % (gate_pp, p2))
print("H3  validation->test Spearman rho=%.2f, p=%.1e" % (rho, p3))
print("H4  crisis median MDD %.1f%% vs 1/N %.1f%%" % (cmed, cN))

rows = [("H1", "Selective prediction (confidence band) beats 1/N out of sample after costs", "Rejected",
         r"only %.1f\%% of configurations beat 1/N on the test period (binomial $p<0.001$)" % (100 * nbeat / n)),
        ("H2", "The regime gate adds value beyond the confidence band alone", "Rejected",
         r"the gate lowers test return by %.0f pp (Wilcoxon $p<0.001$) and inverts transfer" % (-gate_pp)),
        ("H3", "Configuration quality selected on validation transfers to the test period", "Rejected",
         r"validation-to-test rank correlation %.2f (negative at the daily horizon)" % rho),
        ("H4", "Regime-gated abstention protects capital under regime shift (crisis)", "Rejected",
         r"2008-2012: median drawdown %.1f\%% vs 1/N %.1f\%%; the regime gate worsens both" % (cmed, cN))]
print("\n% ---- Table 4 (copy-ready) ----")
print(r"\begin{table}[t]\centering\caption{Study hypotheses and their out-of-sample outcomes.}\label{tab:hyp}")
print(r"\begin{tabularx}{\linewidth}{@{}l >{\raggedright\arraybackslash}X l >{\raggedright\arraybackslash}X@{}}")
print(r"\toprule")
print(r" & Hypothesis (holds if the method works) & Outcome & Evidence \\")
print(r"\midrule")
for h, hyp, out, ev in rows:
    print("%s & %s & %s & %s \\\\" % (h, hyp, out, ev))
print(r"\bottomrule")
print(r"\end{tabularx}\end{table}")

In [ ]:
# fa_t5 - Table 5: EqualWeight-Selected by model x horizon (held-out test)
g = bull.groupby(["algo", "scale"]).agg(TR=("TR", "mean"), AR=("AR", "mean"), MDD=("MDD", "mean")).reset_index()
g["CR"] = g["AR"] / g["MDD"]
g = g.sort_values(["algo", "scale"]).reset_index(drop=True)
disp = g.assign(Model=g.algo.str.upper(), Horizon=g.scale.str.title())[["Model", "Horizon", "TR", "AR", "MDD", "CR"]]
print(disp.to_string(index=False, formatters={"TR": "{:.0f}".format, "AR": "{:.1f}".format,
                                              "MDD": "{:.1f}".format, "CR": "{:.2f}".format}))
print("1/N benchmark   TR %.0f  AR %.1f  MDD %.1f  CR %.2f" % (N1_TR, N1_AR, N1_MDD, N1_CR))

best, worst = g["CR"].idxmax(), g["CR"].idxmin()
def cr_cell(i):
    if i == best:  return r"\textcolor{teal}{%.2f}" % g["CR"][i]
    if i == worst: return r"\textcolor{red}{%.2f}" % g["CR"][i]
    return "%.2f" % g["CR"][i]
print("\n% ---- Table 5 (copy-ready) ----")
print(r"\begin{table}[t]\centering\caption{EqualWeight-Selected by model and horizon (held-out test, 2023--2026). Best Calmar in green, worst in red.}\label{tab:port}")
print(r"\begin{tabular}{llllll}\toprule")
print(r"Model & Horizon & TR \% & AR \% & MDD \% & CR \\")
print(r"\midrule")
for i, r in g.iterrows():
    print("%s & %s & %.0f & %.1f & %.1f & %s \\\\" % (r.algo.upper(), r.scale.title(), r.TR, r.AR, r.MDD, cr_cell(i)))
print(r"1/N benchmark & -- & %.0f & %.1f & %.1f & %.2f \\" % (N1_TR, N1_AR, N1_MDD, N1_CR))
print(r"\bottomrule\end{tabular}\end{table}")

In [ ]:
# fa_t6 - Table 6: crisis test (held-out 2008-2012)
def crow(mask, label):
    s = crisis[mask]
    return (label, tr(s["end_capital_validation"]).mean(), ar(s["end_capital_validation"], YEARS_CRISIS).mean(),
            s["MDD"].mean(), 100 * (s["end_capital_validation"] > s["1/N_validation"]).mean())
cN_tr = crisis["oneN_TR"].iloc[0]; cN_ar = ar(crisis["1/N_validation"].iloc[0], YEARS_CRISIS); cN_mdd = crisis["oneN_MDD"].iloc[0]
r_sel, r_gate = crow(~crisis["gmm"], "Selective (no gate)"), crow(crisis["gmm"], "Regime-gated")
print("EqualWeight 1/N      TR %+.0f  AR %+.1f  MDD %.0f" % (cN_tr, cN_ar, cN_mdd))
for lab, a, b, c, d in (r_sel, r_gate):
    print("%-20s TR %+.0f  AR %+.1f  MDD %.0f  Beat 1/N %.0f%%" % (lab, a, b, c, d))
print("\n% ---- Table 6 (copy-ready) ----")
print(r"\begin{table}[t]\centering\caption{Crisis test (held-out 2008--2012 global financial crisis).}\label{tab:crisis}")
print(r"\begin{tabular}{lllll}\toprule")
print(r"Portfolio (2008-2012) & TR \% & AR \% & MDD \% & Beat 1/N \% \\")
print(r"\midrule")
print(r"EqualWeight 1/N & +%.0f & %.1f & %.0f & -- \\" % (cN_tr, cN_ar, cN_mdd))
for lab, a, b, c, d in (r_sel, r_gate):
    print("%s & %+.0f & %+.1f & %.0f & %.0f \\\\" % (lab, a, b, c, d))
print(r"\bottomrule\end{tabular}\end{table}")

In [ ]:
# Figure 1: total return, drawdown and Calmar-beat by model x horizon
ALG, SC = ["lstm", "xgb", "knn"], ["daily", "weekly", "monthly"]
cell = lambda fn: np.array([[fn(bull[(bull.algo == a) & (bull.scale == s)]) for s in SC] for a in ALG])
panels = [(cell(lambda g: g.TR.mean()),                 "Total return (%)",        "RdYlGn"),
          (cell(lambda g: g.MDD.mean()),                "Maximum drawdown (%)",    "RdYlGn_r"),
          (cell(lambda g: (g.CR > N1_CR).mean() * 100), "Beats 1/N on Calmar (%)", "RdYlGn")]
fig, ax = plt.subplots(1, 3, figsize=(11, 3.4))
for k, (M, t, cm) in enumerate(panels):
    ax[k].imshow(M, cmap=cm, aspect="auto"); ax[k].set_title(t, fontsize=10)
    ax[k].set_xticks(range(3)); ax[k].set_xticklabels([s.title() for s in SC])
    ax[k].set_yticks(range(3)); ax[k].set_yticklabels([a.upper() for a in ALG])
    for i in range(3):
        for j in range(3):
            ax[k].text(j, i, f"{M[i, j]:.0f}", ha="center", va="center")
fig.tight_layout(); save_fig(fig, "fig_grid"); plt.show()

In [ ]:
# Figure 2: test-period return with the regime gate off vs on
fig, a = plt.subplots(figsize=(6, 4.2))
bp = a.boxplot([bull[~bull.gmm].TR, bull[bull.gmm].TR], showfliers=False,
               patch_artist=True, medianprops=dict(color="black"))
for b in bp["boxes"]:
    b.set_facecolor("#dbe6ef")
a.set_xticks([1, 2]); a.set_xticklabels(["Selective (no gate)", "Regime-gated"])
a.axhline(N1_TR, ls="--", color="#d62728"); a.text(2.32, N1_TR + 4, "1/N", color="#d62728")
a.set_ylabel("Total return (%), test period")
a.set_title("Regime gate reduces out-of-sample return", color="#555")
fig.tight_layout(); save_fig(fig, "fig_gmm"); plt.show()

In [ ]:
# Figure 3: mean total return against the holding count K
ks = sorted(bull.k.unique())
test = [bull[bull.k == k].TR.mean() for k in ks]
vald = [tr(bull[bull.k == k].end_capital_train).mean() for k in ks]
fig, a = plt.subplots(figsize=(7, 4.2))
a.plot(ks, vald, color="#9a9a9a", label="Validation (2021-23, selection)")
a.plot(ks, test, color="#2c6fbb", lw=2, label="Test (2023-26, held-out)")
a.axhline(N1_TR, ls="--", color="#d62728"); a.text(37, N1_TR + 4, "1/N", color="#d62728")
a.set_ylim(0, 160); a.set_xlabel("Holding count K"); a.set_ylabel("Mean total return (%)")
a.set_title("No holding count beats 1/N out of sample", color="#555"); a.legend()
fig.tight_layout(); save_fig(fig, "fig_K"); plt.show()

In [ ]:
# Figure 4: risk-return of all configurations (test)
dom = bull[(bull.end_capital_validation > bull["1/N_validation"]) & (bull.mdd_validation < N1_MDD)]
ms = bull[(bull.scale == "monthly") & (~bull.gmm)]
ms = ms.loc[ms.groupby(["algo", "fusion", "tier", "train_start", "x_days"]).end_capital_train.idxmax()]
fig, a = plt.subplots(figsize=(7, 5))
a.scatter(bull.MDD, bull.TR, s=6, color="#c9c9c9", alpha=.5, label="All configurations")
a.scatter(dom.MDD, dom.TR, s=10, color="#2ca02c", label="Dominate 1/N (%.1f%%)" % (100 * len(dom) / len(bull)))
a.scatter(ms.MDD, ms.TR, s=14, color="#c0392b", label="Monthly selective (validation-selected)")
a.scatter([N1_MDD], [N1_TR], marker="D", s=60, color="black"); a.text(N1_MDD + .5, N1_TR + 3, "1/N")
a.set_xlim(0, 45); a.set_ylim(-30, 260)
a.set_xlabel("Maximum drawdown (%), test period"); a.set_ylabel("Total return (%), test period")
a.set_title("Risk and return of all configurations, held-out test period", color="#555"); a.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "fig_frontier"); plt.show()

In [ ]:
# Figure 5: risk-return over the 2008-2012 crisis
cN, cNm = crisis.oneN_TR.iloc[0], crisis.oneN_MDD.iloc[0]
fig, a = plt.subplots(figsize=(7, 5))
a.scatter(crisis[~crisis.gmm].MDD, crisis[~crisis.gmm].TR, s=6, color="#2c6fbb", alpha=.45, label="Selective (no gate)")
a.scatter(crisis[crisis.gmm].MDD,  crisis[crisis.gmm].TR,  s=6, color="#c0392b", alpha=.45, label="Regime-gated")
a.scatter([cNm], [cN], marker="D", s=60, color="black"); a.text(cNm + 1, cN + 8, "1/N")
a.set_xlabel("Maximum drawdown (%), 2008-2012"); a.set_ylabel("Total return (%), 2008-2012")
a.set_title("Crisis test: no configuration protected capital", color="#555"); a.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "fig_crisis"); plt.show()